In [22]:
import pandas as pd
data = pd.read_csv('metadata.csv').sample(n=200).set_index('video_id').to_dict('index')

In [ ]:
from pathlib import Path

import librosa
from tqdm.auto import tqdm


for id, row in tqdm(data.items()):
    video_path = Path('/home/andrey/onti') / row['path']
    files = list(video_path.glob('*.m4a'))
    if len(files) == 0:
        continue
    path = files[0]
    waveform, sr = librosa.load(path, sr=16_000)
    row['audio'] = {'array': waveform, 'sampling_rate': sr}

In [28]:
data = {
    id: row for id, row in data.items()
    if 'audio' in row
}
len(data)

170

In [30]:
for id, row in data.items():
    arr = row['audio']['array']
    row['audio']['array'] = (
        arr[16_000*60*5:16_000*60*6]
        if len(arr) >= 16_000*60*6
        else arr[16_000*60*1]
    )

In [40]:
for id, row in data.items():
    row['video_id'] = id

In [41]:
from datasets import Dataset, Features, Value, Audio

dataset = Dataset.from_list(list(data.values()), features=Features({ # type: ignore
    'audio': Audio(decode=True),
    'video_id': Value('string'),
    'video_title': Value('string'),
    'video_desc': Value('string'),
    'video_duration': Value('float32'),
    'speaker': Value('string'),
    'video_date': Value('string'),
    'status': Value('string'),
    'project_id': Value('string'),
    'project_name': Value('string'),
    'folder_id': Value('string'),
    'folder_name': Value('string'),
    'path': Value('string'),
}))

In [42]:
dataset.save_to_disk('/asr_datasets/ontico_unlabeled')

Saving the dataset (0/1 shards):   0%|          | 0/170 [00:00<?, ? examples/s]